In [1]:
from google.colab import drive, userdata
from pathlib import Path
import os, subprocess, json, time

drive.mount("/content/drive")

PROJECT_ROOT = Path("/content/drive/MyDrive/kride-track-b")
PROJECT_ROOT.mkdir(parents=True, exist_ok=True)

WORK = Path("/content/kride_work")
WORK.mkdir(parents=True, exist_ok=True)

print("PROJECT_ROOT:", PROJECT_ROOT)
print("WORK:", WORK)
subprocess.run(["bash", "-lc", "nvidia-smi || true"], check=False)

Mounted at /content/drive
PROJECT_ROOT: /content/drive/MyDrive/kride-track-b
WORK: /content/kride_work


CompletedProcess(args=['bash', '-lc', 'nvidia-smi || true'], returncode=0)

In [2]:
import os
from google.colab import userdata

DAGSHUB_USERNAME = "myelin24m"
DAGSHUB_REPO = "Kride"
DAGSHUB_TOKEN = userdata.get("DAGSHUB_TOKEN")

os.environ["DAGSHUB_USER"] = DAGSHUB_USERNAME
os.environ["DAGSHUB_REPO"] = DAGSHUB_REPO
os.environ["DAGSHUB_TOKEN"] = DAGSHUB_TOKEN
os.environ["MLFLOW_TRACKING_USERNAME"] = DAGSHUB_USERNAME
os.environ["MLFLOW_TRACKING_PASSWORD"] = DAGSHUB_TOKEN
os.environ["MLFLOW_TRACKING_URI"] = f"https://dagshub.com/{DAGSHUB_USERNAME}/{DAGSHUB_REPO}.mlflow"

print("MLflow URI:", os.environ["MLFLOW_TRACKING_URI"])

MLflow URI: https://dagshub.com/myelin24m/Kride.mlflow


In [3]:
!apt-get update -qq
!apt-get install -y -qq ffmpeg git curl libosmesa6 libosmesa6-dev freeglut3-dev libglfw3-dev

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Selecting previously unselected package freeglut3:amd64.
(Reading database ... 122363 files and directories currently installed.)
Preparing to unpack .../00-freeglut3_2.8.1-6_amd64.deb ...
Unpacking freeglut3:amd64 (2.8.1-6) ...
Selecting previously unselected package libglx-dev:amd64.
Preparing to unpack .../01-libglx-dev_1.4.0-1_amd64.deb ...
Unpacking libglx-dev:amd64 (1.4.0-1) ...
Selecting previously unselected package libgl-dev:amd64.
Preparing to unpack .../02-libgl-dev_1.4.0-1_amd64.deb ...
Unpacking libgl-dev:amd64 (1.4.0-1) ...
Selecting previously unselected package libglvnd-core-dev:amd64.
Preparing to unpack .../03-libglvnd-core-dev_1.4.0-1_amd64.deb ...
Unpacking libglvnd-core-dev:amd64 (1.4.0-1) ...
Selecting previously unselected package libegl-dev:amd64.
Preparing to unpack .../04-li

In [4]:
from pathlib import Path
import os

WORK_DIR = Path("/content/kride_work/AnimatedDrawings")

if not WORK_DIR.exists():
    !git clone https://github.com/facebookresearch/AnimatedDrawings.git /content/kride_work/AnimatedDrawings

anno = WORK_DIR / "examples" / "annotations_to_animation.py"
src = anno.read_text()
if "'view': {'USE_MESA': True" not in src and '"view": {"USE_MESA": True' not in src:
    src = src.replace(
        "'controller': {",
        "'view': {'USE_MESA': True},\n        'controller': {",
        1,
    )
    anno.write_text(src)
    print("USE_MESA patch 완료")

Cloning into '/content/kride_work/AnimatedDrawings'...
remote: Enumerating objects: 1051, done.
remote: Counting objects: 100% (383/383), done.
remote: Compressing objects: 100% (168/168), done.
remote: Total 1051 (delta 288), reused 215 (delta 215), pack-reused 668 (from 2)
Receiving objects: 100% (1051/1051), 110.35 MiB | 37.15 MiB/s, done.
Resolving deltas: 100% (530/530), done.
USE_MESA patch 완료


In [6]:
from pathlib import Path

CONDA_DIR = "/content/miniconda"
ENV_DIR = f"{CONDA_DIR}/envs/ad_env"
PYTHON_EXE = f"{ENV_DIR}/bin/python"
PIP_EXE = f"{ENV_DIR}/bin/pip"
TORCHSERVE = f"{ENV_DIR}/bin/torchserve"

if not Path(ENV_DIR).exists():
    !/content/miniconda/bin/conda create -n ad_env python=3.8 -y
else:
    print("✅ ad_env already exists")

print("python exists:", Path(PYTHON_EXE).exists())

✅ ad_env already exists
python exists: True


In [7]:
from pathlib import Path
import subprocess, textwrap

PIP_EXE = "/content/miniconda/envs/ad_env/bin/pip"
PYTHON_EXE = "/content/miniconda/envs/ad_env/bin/python"

assert Path(PIP_EXE).exists(), f"pip 없음: {PIP_EXE}"
assert Path(PYTHON_EXE).exists(), f"python 없음: {PYTHON_EXE}"

def run(cmd, title):
    print(f"\n▶️ {title}")
    print(" ".join(cmd))
    result = subprocess.run(cmd, text=True)
    if result.returncode != 0:
        raise RuntimeError(f"실패: {title}")

run([PIP_EXE, "install", "--upgrade", "pip", "setuptools==65.5.0", "wheel", "six", "--quiet"], "기본 빌드 도구 설치")

run([PIP_EXE, "install", "chumpy", "--no-build-isolation", "--quiet"], "chumpy 설치")

run([
    PIP_EXE, "install",
    "torch==1.13.1+cu116",
    "torchvision==0.14.1+cu116",
    "torchaudio==0.13.1",
    "--extra-index-url", "https://download.pytorch.org/whl/cu116",
    "--quiet",
], "PyTorch 1.13.1 CUDA 11.6 설치")

run([
    PIP_EXE, "install",
    "mmcv-full==1.7.0",
    "-f", "https://download.openmmlab.com/mmcv/dist/cu116/torch1.13.0/index.html",
    "--quiet",
], "mmcv-full 설치")

run([PIP_EXE, "install", "mmdet==2.28.2", "mmpose==0.28.1", "--quiet"], "mmdet / mmpose 설치")

run([PIP_EXE, "install", "torchserve==0.8.0", "torch-model-archiver==0.8.0", "--quiet"], "TorchServe 설치")

run([
    PIP_EXE, "install",
    "scikit-image",
    "scikit-learn",
    "PyOpenGL==3.1.7",
    "PyOpenGL-accelerate",
    "imageio",
    "imageio-ffmpeg",
    "shapely",
    "pillow",
    "pyyaml",
    "requests",
    "mlflow",
    "dagshub",
    "tqdm",
    "matplotlib",
    "opencv-python-headless",
    "--quiet",
], "렌더링 보조 패키지 설치")

print("\n✅ AnimatedDrawings ad_env 의존성 설치 완료")


▶️ 기본 빌드 도구 설치
/content/miniconda/envs/ad_env/bin/pip install --upgrade pip setuptools==65.5.0 wheel six --quiet

▶️ chumpy 설치
/content/miniconda/envs/ad_env/bin/pip install chumpy --no-build-isolation --quiet

▶️ PyTorch 1.13.1 CUDA 11.6 설치
/content/miniconda/envs/ad_env/bin/pip install torch==1.13.1+cu116 torchvision==0.14.1+cu116 torchaudio==0.13.1 --extra-index-url https://download.pytorch.org/whl/cu116 --quiet

▶️ mmcv-full 설치
/content/miniconda/envs/ad_env/bin/pip install mmcv-full==1.7.0 -f https://download.openmmlab.com/mmcv/dist/cu116/torch1.13.0/index.html --quiet

▶️ mmdet / mmpose 설치
/content/miniconda/envs/ad_env/bin/pip install mmdet==2.28.2 mmpose==0.28.1 --quiet

▶️ TorchServe 설치
/content/miniconda/envs/ad_env/bin/pip install torchserve==0.8.0 torch-model-archiver==0.8.0 --quiet

▶️ 렌더링 보조 패키지 설치
/content/miniconda/envs/ad_env/bin/pip install scikit-image scikit-learn PyOpenGL==3.1.7 PyOpenGL-accelerate imageio imageio-ffmpeg shapely pillow pyyaml requests mlflow dagsh

In [8]:
import subprocess

PYTHON_EXE = "/content/miniconda/envs/ad_env/bin/python"
TORCHSERVE = "/content/miniconda/envs/ad_env/bin/torchserve"

print(subprocess.check_output([PYTHON_EXE, "--version"], text=True))
print(subprocess.check_output([TORCHSERVE, "--version"], text=True))

check_code = """
import torch, torchvision, torchaudio
import mmcv
import mmdet
import mmpose
import OpenGL
print("torch:", torch.__version__)
print("torchvision:", torchvision.__version__)
print("torchaudio:", torchaudio.__version__)
print("mmcv:", mmcv.__version__)
print("mmdet:", mmdet.__version__)
print("mmpose:", mmpose.__version__)
print("OpenGL ok")
"""

print(subprocess.check_output([PYTHON_EXE, "-c", check_code], text=True))

Python 3.8.20

TorchServe Version is 0.8.0

torch: 1.13.1+cu116
torchvision: 0.14.1+cu116
torchaudio: 0.13.1+cu116
mmcv: 1.7.0
mmdet: 2.28.2
mmpose: 0.28.1
OpenGL ok



In [ ]:
from pathlib import Path
import shutil

SRC = Path("/content/kride_work/six_doodle_actions_fallback/combined_sequential.mp4")
DST = Path("/content/drive/MyDrive/kride-track-b/combined_sequential.mp4")

if SRC.exists():
    shutil.copy2(SRC, DST)
    print("saved:", DST)
else:
    print("missing:", SRC)d

In [10]:
from pathlib import Path

ROOT_OUTPUT = Path("/content/kride_work/six_doodle_actions_fallback")

print("ROOT exists:", ROOT_OUTPUT.exists())
print("video.gif count:", len(list(ROOT_OUTPUT.rglob("video.gif"))))
print("combined mp4 exists:", (ROOT_OUTPUT / "combined_sequential.mp4").exists())

for p in list(ROOT_OUTPUT.rglob("video.gif"))[:10]:
    print(p)

ROOT exists: False
video.gif count: 0
combined mp4 exists: False


In [11]:
import os, time, subprocess, shutil, json, re
from pathlib import Path
from IPython.display import Video, display

WORK_DIR = Path("/content/kride_work/AnimatedDrawings")
PYTHON_EXE = "/content/miniconda/envs/ad_env/bin/python"
ROOT_OUTPUT = Path("/content/kride_work/six_doodle_actions_fallback")
ROOT_OUTPUT.mkdir(parents=True, exist_ok=True)

def p(*parts):
    return str(WORK_DIR.joinpath(*parts))

CHAR = {
    "char1": p("examples", "characters", "char1"),
    "char2": p("examples", "characters", "char2"),
    "char3": p("examples", "characters", "char3"),
    "char4": p("examples", "characters", "char4"),
    "char5": p("examples", "characters", "char5"),
    "char6": p("examples", "characters", "char6"),
}

MOTION = {
    "dab": p("examples", "config", "motion", "dab.yaml"),
    "wave_hello": p("examples", "config", "motion", "wave_hello.yaml"),
    "jumping": p("examples", "config", "motion", "jumping.yaml"),
    "jumping_jacks": p("examples", "config", "motion", "jumping_jacks.yaml"),
    "zombie": p("examples", "config", "motion", "zombie.yaml"),
    "jesse_dance": p("examples", "config", "motion", "jesse_dance.yaml"),
}

RETARGET = {
    "fair1_ppf": p("examples", "config", "retarget", "fair1_ppf.yaml"),
    "cmu1_pfp": p("examples", "config", "retarget", "cmu1_pfp.yaml"),
    "mixamo_fff": p("examples", "config", "retarget", "mixamo_fff.yaml"),
}

env = os.environ.copy()
env["PYOPENGL_PLATFORM"] = "osmesa"
env["MESA_GL_VERSION_OVERRIDE"] = "3.3"
env["MESA_GLSL_VERSION_OVERRIDE"] = "330"
env["LIBGL_ALWAYS_SOFTWARE"] = "1"
env["MPLBACKEND"] = "Agg"
env["PYTHONPATH"] = str(WORK_DIR)

def copy_character(src_char_dir, dst_dir):
    dst = Path(dst_dir)
    if dst.exists():
        shutil.rmtree(dst)
    shutil.copytree(src_char_dir, dst)
    return dst

def render_one(index, character, motion, retarget):
    action = Path(motion).stem
    out_dir = ROOT_OUTPUT / f"{index:02d}_{action}"
    copy_character(character, out_dir)

    cmd = [
        PYTHON_EXE,
        str(WORK_DIR / "examples" / "annotations_to_animation.py"),
        str(out_dir),
        motion,
        retarget,
    ]

    result = subprocess.run(
        cmd,
        env=env,
        cwd=str(WORK_DIR),
        capture_output=True,
        text=True,
    )

    (out_dir / "render_stdout.txt").write_text(result.stdout[-10000:])
    (out_dir / "render_stderr.txt").write_text(result.stderr[-10000:])

    gif_path = out_dir / "video.gif"
    print(f"{index}. {action} return={result.returncode} gif={gif_path.exists()}")

    if not gif_path.exists():
        print(result.stderr[-800:])

    return {
        "index": index,
        "action": action,
        "gif_path": str(gif_path) if gif_path.exists() else None,
        "return_code": result.returncode,
        "out_dir": str(out_dir),
    }

def make_sequential_mp4(items, output_path):
    valid = [item for item in items if item["gif_path"] and Path(item["gif_path"]).exists()]
    if not valid:
        raise RuntimeError("생성된 GIF가 없습니다.")

    segments_dir = ROOT_OUTPUT / "mp4_segments"
    if segments_dir.exists():
        shutil.rmtree(segments_dir)
    segments_dir.mkdir(parents=True, exist_ok=True)

    segment_paths = []

    for item in valid:
        segment_path = segments_dir / f"{item['index']:02d}_{item['action']}.mp4"

        vf = (
            "fps=12,"
            "scale=512:512:force_original_aspect_ratio=decrease,"
            "pad=512:512:(ow-iw)/2:(oh-ih)/2:color=white,"
            "format=yuv420p"
        )

        cmd = [
            "ffmpeg", "-y",
            "-i", item["gif_path"],
            "-vf", vf,
            "-an",
            "-c:v", "libx264",
            "-preset", "veryfast",
            "-crf", "23",
            str(segment_path),
        ]

        result = subprocess.run(cmd, capture_output=True, text=True)
        if result.returncode == 0 and segment_path.exists():
            segment_paths.append(segment_path)
            print("✅ segment:", segment_path)
        else:
            print("❌ segment 실패:", item["gif_path"])
            print(result.stderr[-800:])

    concat_file = segments_dir / "concat.txt"
    concat_file.write_text(
        "\n".join([f"file '{path.as_posix()}'" for path in segment_paths]),
        encoding="utf-8",
    )

    cmd_concat = [
        "ffmpeg", "-y",
        "-f", "concat",
        "-safe", "0",
        "-i", str(concat_file),
        "-c", "copy",
        str(output_path),
    ]

    result = subprocess.run(cmd_concat, capture_output=True, text=True)

    if result.returncode != 0:
        cmd_concat = [
            "ffmpeg", "-y",
            "-f", "concat",
            "-safe", "0",
            "-i", str(concat_file),
            "-c:v", "libx264",
            "-pix_fmt", "yuv420p",
            "-preset", "veryfast",
            "-crf", "23",
            str(output_path),
        ]
        result = subprocess.run(cmd_concat, capture_output=True, text=True)

    if not output_path.exists():
        print(result.stderr[-1000:])
        raise RuntimeError("combined_sequential.mp4 생성 실패")

    return output_path

jobs = [
    (CHAR["char1"], MOTION["dab"], RETARGET["fair1_ppf"]),
    (CHAR["char2"], MOTION["wave_hello"], RETARGET["fair1_ppf"]),
    (CHAR["char3"], MOTION["jumping"], RETARGET["fair1_ppf"]),
    (CHAR["char4"], MOTION["jumping_jacks"], RETARGET["cmu1_pfp"]),
    (CHAR["char5"], MOTION["zombie"], RETARGET["fair1_ppf"]),
    (CHAR["char6"], MOTION["jesse_dance"], RETARGET["mixamo_fff"]),
]

results = []
for i, job in enumerate(jobs, 1):
    results.append(render_one(i, *job))

summary_path = ROOT_OUTPUT / "render_summary.json"
summary_path.write_text(json.dumps(results, ensure_ascii=False, indent=2), encoding="utf-8")

combined_mp4 = make_sequential_mp4(results, ROOT_OUTPUT / "combined_sequential.mp4")

print("🎬 combined MP4:", combined_mp4)
display(Video(str(combined_mp4), embed=True, width=520))

1. dab return=0 gif=True
2. wave_hello return=0 gif=True
3. jumping return=0 gif=True
4. jumping_jacks return=0 gif=True
5. zombie return=1 gif=False
/AnimatedDrawings/examples/annotations_to_animation.py", line 41, in annotations_to_animation
    animated_drawings.render.start(output_mvc_cfn_fn)
  File "/content/kride_work/AnimatedDrawings/animated_drawings/render.py", line 21, in start
    scene = Scene(cfg.scene)
  File "/content/kride_work/AnimatedDrawings/animated_drawings/model/scene.py", line 30, in __init__
    ad = AnimatedDrawing(*each)
  File "/content/kride_work/AnimatedDrawings/animated_drawings/model/animated_drawing.py", line 247, in __init__
    self._modify_retargeting_cfg_for_character()
  File "/content/kride_work/AnimatedDrawings/animated_drawings/model/animated_drawing.py", line 284, in _modify_retargeting_cfg_for_character
    assert False, msg
AssertionError: Could not find joint1 in runtime check: right_shoulder

6. jesse_dance return=1 gif=False
/AnimatedDrawin

In [12]:
from pathlib import Path
import shutil

SRC = Path("/content/kride_work/six_doodle_actions_fallback/combined_sequential.mp4")
DST = Path("/content/drive/MyDrive/kride-track-b/combined_sequential.mp4")

assert SRC.exists(), f"missing: {SRC}"

DST.parent.mkdir(parents=True, exist_ok=True)
shutil.copy2(SRC, DST)

print("✅ Drive 저장 완료:", DST)

✅ Drive 저장 완료: /content/drive/MyDrive/kride-track-b/combined_sequential.mp4
